In [1]:
import os
import gc
import random
import numpy as np
import pandas as pd

import torch
from datasets import Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())


/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


torch: 2.0.0
cuda available: True


In [2]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)


# 1. 路径配置

In [3]:
# Base model
model_checkpoint = "/kaggle/input/distilroberta-base/distilroberta-base"

# Core datasets from original pipeline
PATH_EXTERNAL = "/kaggle/input/daigt-external-dataset/daigt_external_dataset.csv"
PATH_LLM7 = "/kaggle/input/llm-7-prompt-training-dataset/train_essays_RDizzl3_seven_v1.csv"
PATH_FB21 = "/kaggle/input/feedback21-and-fb-effectiveness/feedback-prize-2021.csv"
PATH_FPE = "/kaggle/input/feedback21-and-fb-effectiveness/feedback-prize-effectiveness.csv"
PATH_H2O = "/kaggle/input/h2oai-predict-the-llm/train.csv"
PATH_DRCAT2 = "/kaggle/input/daigt-v2-train-dataset/train_v2_drcat_02.csv"
PATH_PALM = "/kaggle/input/llm-generated-essay-using-palm-from-google-gen-ai/LLM_generated_essay_PaLM.csv"
PATH_MISTRAL = "/kaggle/input/llm-mistral-7b-instruct-texts/Mistral7B_CME_v7_15_percent_corruption.csv"
PATH_LLAMA = "/kaggle/input/daigt-data-llama-70b-and-falcon180b/llama_falcon_v3.csv"
PATH_CLAUDE = "/kaggle/input/hello-claude-1000-essays-from-anthropic/persuade15_claude_instant1.csv"

# Competition files
PATH_TEST = "/kaggle/input/llm-detect-ai-generated-text/test_essays.csv"
PATH_SAMPLE_SUB = "/kaggle/input/llm-detect-ai-generated-text/sample_submission.csv"

# Output
OUTPUT_DIR = "/kaggle/working/distilroberta-finetuned_v927"
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 2. 数据构建

In [4]:
# 2.1 External dataset -> source_text:0, text:1
df_ai = pd.read_csv(PATH_EXTERNAL)
print('external rows:', len(df_ai))

df_ai = df_ai.sample(frac=1, random_state=42)
df_train = df_ai[:-200]
df_valid = df_ai[-200:]

t1 = pd.DataFrame({'text': df_train.source_text, 'label': 0})
t2 = pd.DataFrame({'text': df_train.text, 'label': 1})
train_ext = pd.concat([t1, t2], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)

v1 = pd.DataFrame({'text': df_valid.source_text, 'label': 0})
v2 = pd.DataFrame({'text': df_valid.text, 'label': 1})
valid_ext = pd.concat([v1, v2], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)

print('train_ext:', train_ext.shape, 'valid_ext:', valid_ext.shape)


external rows: 2421
train_ext: (4442, 2) valid_ext: (400, 2)


In [5]:
# 2.2 LLM 7 prompt (label invert)
llm7prompt = pd.read_csv(PATH_LLM7)
llm7prompt['label'] = llm7prompt['label'].apply(lambda x: 1 if x == 0 else 0)
llm7prompt = llm7prompt[['text', 'label']]

# 2.3 Feedback 2021 + FPE -> human samples (label 0), length filter
feedback_1 = pd.read_csv(PATH_FB21)
feedback_1['chars_count'] = feedback_1.text.astype(str).apply(len)
feedback_2 = pd.read_csv(PATH_FPE)
feedback_2['chars_count'] = feedback_2.text.astype(str).apply(len)

A = feedback_1[(feedback_1['chars_count'] > 1000) & (feedback_1['chars_count'] < 1600)][['text']]
B = feedback_2[(feedback_2['chars_count'] > 1000) & (feedback_2['chars_count'] < 1600)][['text']]
C = pd.concat([A, B], axis=0).reset_index(drop=True)
C['label'] = 0

# 2.4 H2O non-null response -> label 1
h2o_raw = pd.read_csv(PATH_H2O)
h2o = h2o_raw[np.invert(h2o_raw.Response.isnull())].rename(columns={'Response': 'text'})[['text']]
h2o['label'] = 1
h2o.text = h2o.text.fillna('')

# 2.5 drcat v2 (label invert + dedup)
train2 = pd.read_csv(PATH_DRCAT2, sep=',')
train2 = train2.drop_duplicates(subset=['text'])
train2['label'] = train2['label'].apply(lambda x: 1 if x == 0 else 0)
train2 = train2[['text', 'label']]

print('llm7prompt:', len(llm7prompt))
print('C(feedback):', len(C))
print('h2o:', len(h2o))
print('train2:', len(train2))


llm7prompt: 15871
C(feedback): 4725
h2o: 3969
train2: 44868


In [11]:
# 2.6 assemble train0 (first merge)
train0 = pd.concat([
    llm7prompt,
    train_ext,
    train2,
    C,
    h2o,
    valid_ext,
], axis=0).reset_index(drop=True).drop_duplicates().sample(frac=1, random_state=42).reset_index(drop=True)[['text', 'label']]

# 2.7 additional llm/human-like sets
palm = pd.read_csv(PATH_PALM)[['text']]
palm['label'] = 0

mistral = pd.read_csv(PATH_MISTRAL)[['text']]
mistral['label'] = 0

llama = pd.read_csv(PATH_LLAMA)[['text']]
llama['label'] = 0

claude = pd.read_csv(PATH_CLAUDE)[['essay_text']]
claude = claude.rename(columns={'essay_text': 'text'})
claude['label'] = 0

train0 = pd.concat([train0, claude, mistral, llama, palm], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
train0 = train0.drop_duplicates(subset=['text']).reset_index(drop=True)

train0['text'] = train0['text'].fillna('').astype(str).str.strip('')

print('train0 size:', len(train0))
print('label distribution:')
print(train0.label.value_counts())


train0 size: 77865
label distribution:
label
1    42238
0    35627
Name: count, dtype: int64


## 3. KFold切分（10折后取第1折）

In [12]:
sk = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
for i, (tr, val) in enumerate(sk.split(train0, train0.label)):
    train = train0.iloc[tr].copy()
    valid = train0.iloc[val].copy()
    print('use fold:', i)
    break

print('train:', train.shape, train.label.value_counts().to_dict())
print('valid:', valid.shape, valid.label.value_counts().to_dict())


use fold: 0
train: (70078, 2) {1: 38014, 0: 32064}
valid: (7787, 2) {1: 4224, 0: 3563}


## 4. Tokenizer与Dataset

In [14]:
from transformers import AutoTokenizer
from datasets import Dataset
from datasets.utils.logging import disable_progress_bar

disable_progress_bar()

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    return tokenizer(
        examples["text"],
        max_length=128,
        padding="max_length",
        truncation=True
    )

train["text"] = train["text"].fillna("").astype(str).str.strip()
valid["text"] = valid["text"].fillna("").astype(str).str.strip()

ds_train = Dataset.from_pandas(train[["text", "label"]], preserve_index=False)
ds_valid = Dataset.from_pandas(valid[["text", "label"]], preserve_index=False)

ds_train_enc = ds_train.map(preprocess_function, batched=True)
ds_valid_enc = ds_valid.map(preprocess_function, batched=True)

print(ds_train_enc[0])
print(ds_valid_enc[0])

{'text': 'Summer projects should be student-designed because students are busy during the summer and teachers make projects to complicated.\n\nOne of the reasons why they should student-designed is that students are busy during the summer. Students already go through so much during the school year, whether it be drama with another student, annoying teachers, bullying, and even fights, so when school is over students just want to get away from it all. Most of the time students will go see a parent, or go out of town, so they wonÃ\x82Â´t even have time to do a project.\n\nMy other reason is that teachers make projects to complicated. Most teachers give projects at least twice a year, and one of the two will always be to complicated to do. Teachers know that students go out of town during the summer, but they will still give you a super complicated project just because youÃ\x82Â´ll have the whole summer to do it.\n\nSome people might think that summer projects should be teacher-designed b

## 5. 训练配置

In [15]:
num_labels = 2
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model.to(device)

metric_name = 'roc_auc'
model_name = 'distilroberta'
batch_size = 2
num_train_epochs = 16.0

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    lr_scheduler_type='cosine',
    optim='adamw_torch',
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=8,
    num_train_epochs=num_train_epochs,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model=metric_name,
    report_to='none',
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    seed=42,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    x = logits - np.max(logits, axis=-1, keepdims=True)
    probs = np.exp(x) / np.sum(np.exp(x), axis=-1, keepdims=True)
    auc = roc_auc_score(labels, probs[:, 1], multi_class='ovr')
    return {'roc_auc': auc}

early_stopping = EarlyStoppingCallback(early_stopping_patience=2)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train_enc,
    eval_dataset=ds_valid_enc,
    tokenizer=tokenizer,
    callbacks=[early_stopping],
    compute_metrics=compute_metrics,
)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at /kaggle/input/distilroberta-base/distilroberta-base and are newly initialized: ['classifier.out_proj.bias', 'classifier.out_proj.weight', 'classifier.dense.weight', 'classifier.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 6. 开始训练并保存

In [16]:
train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print('model saved to:', OUTPUT_DIR)

# 保存训练日志
log_history = pd.DataFrame(trainer.state.log_history)
log_path = os.path.join(OUTPUT_DIR, 'log_history.csv')
log_history.to_csv(log_path, index=False)
print('log_history saved:', log_path)


You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Roc Auc
0,0.056800,0.098784,0.998722
1,0.042400,0.085852,0.999052
2,0.037000,0.086334,0.998829
3,0.032000,0.049933,0.999024


model saved to: /kaggle/working/distilroberta-finetuned_v927
log_history saved: /kaggle/working/distilroberta-finetuned_v927/log_history.csv


## 7. 本折验证集AUC（最终模型）

In [17]:
pred_valid = trainer.predict(ds_valid_enc).predictions
x = pred_valid - np.max(pred_valid, axis=-1, keepdims=True)
probs_valid = np.exp(x) / np.sum(np.exp(x), axis=-1, keepdims=True)
auc_valid = roc_auc_score(valid['label'].values, probs_valid[:, 1])
print('valid ROC-AUC:', auc_valid)

valid_out = valid[['text', 'label']].copy()
valid_out['pred_score_cls1'] = probs_valid[:, 1]
valid_out.to_csv(os.path.join(OUTPUT_DIR, 'valid_pred_cls1.csv'), index=False)


valid ROC-AUC: 0.999051634964577


## 8. 测试集推理并导出 submission / sub_nn

In [18]:
import os
import numpy as np
import pandas as pd
from datasets import Dataset

# 1. 读取测试集与提交模板
test_df = pd.read_csv(PATH_TEST)
sample_sub = pd.read_csv(PATH_SAMPLE_SUB)

# 2. 清洗文本
test_df["text"] = test_df["text"].fillna("").astype(str).str.strip()

# 3. 转为 Hugging Face Dataset
ds_test = Dataset.from_pandas(test_df[["id", "text"]], preserve_index=False)

# 4. 定义测试集预处理函数
def preprocess_test_function(examples):
    return tokenizer(
        examples["text"],
        max_length=128,
        padding="max_length",
        truncation=True
    )

# 5. 编码测试集
ds_test_enc = ds_test.map(preprocess_test_function, batched=True)

# 6. 删除原始文本列，只保留模型需要的输入列和id
#    id 留着方便后面生成提交文件
ds_test_enc.set_format(type="torch", columns=["input_ids", "attention_mask"])

# 7. 预测
pred_output = trainer.predict(ds_test_enc)
logits_test = pred_output.predictions

# 8. softmax 转概率
x = logits_test - np.max(logits_test, axis=-1, keepdims=True)
probs_test = np.exp(x) / np.sum(np.exp(x), axis=-1, keepdims=True)

# 9. 二分类默认取第 1 类概率作为 generated
positive_class_index = 1
pred = probs_test[:, positive_class_index]

# 10. 生成神经网络原始提交文件
sub_nn = pd.DataFrame({
    "id": test_df["id"],
    "generated": pred
})

sub_nn_path = os.path.join(OUTPUT_DIR, "sub_nn.csv")
sub_nn.to_csv(sub_nn_path, index=False)
print("saved:", sub_nn_path)

# 11. 按 sample submission 格式生成最终提交文件
submission = sample_sub[["id"]].merge(sub_nn, on="id", how="left")

sub_path = os.path.join(OUTPUT_DIR, "submission.csv")
submission.to_csv(sub_path, index=False)
print("saved:", sub_path)

# 12. 预览前几行
print(submission.head())

saved: /kaggle/working/distilroberta-finetuned_v927/sub_nn.csv
saved: /kaggle/working/distilroberta-finetuned_v927/submission.csv
         id  generated
0  0000aaaa   0.992837
1  1111bbbb   0.995826
2  2222cccc   0.991067


## 9. 产物清单

In [19]:
for p in [
    OUTPUT_DIR,
    os.path.join(OUTPUT_DIR, 'sub_nn.csv'),
    os.path.join(OUTPUT_DIR, 'submission.csv'),
    os.path.join(OUTPUT_DIR, 'log_history.csv'),
    os.path.join(OUTPUT_DIR, 'valid_pred_cls1.csv'),
]:
    print(p, '->', os.path.exists(p))

gc.collect()


/kaggle/working/distilroberta-finetuned_v927 -> True
/kaggle/working/distilroberta-finetuned_v927/sub_nn.csv -> True
/kaggle/working/distilroberta-finetuned_v927/submission.csv -> True
/kaggle/working/distilroberta-finetuned_v927/log_history.csv -> True
/kaggle/working/distilroberta-finetuned_v927/valid_pred_cls1.csv -> True


61